#Set up




```
# Dit is opgemaakt als code
```

**Authorise colab to connect to google drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Libraries**

In [ ]:
pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

#for checking results
import statsmodels.tsa.api as tsa
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.panel import RandomEffects
from scipy.stats import chi2

#Data Import

construct a file called Ectr3_data with the assignment data in it in your drive for the path to work

In [ ]:
file_path = '/content/drive/MyDrive/Ectr3_data/data_assignment1.csv'
df = pd.read_csv(file_path)

In [ ]:
#data structure
df.head()

,Unnamed: 0,X,exp,wks,bluecol,ind,south,smsa,married,gender,union,edu,col,lwage
0,1,1,3,32,0,0,1,0,1,0,0,9,0,5.56068
1,2,2,4,43,0,0,1,0,1,0,0,9,0,5.72031
2,3,3,5,40,0,0,1,0,1,0,0,9,0,5.99645
3,4,4,6,39,0,0,1,0,1,0,0,9,0,5.99645
4,5,5,7,42,0,1,1,0,1,0,0,9,0,6.06146


In [ ]:
#drop redundant columns
df.drop(columns=['Unnamed: 0', 'X'], inplace=True)

#construct individual and year indicators
N = 595
T = 7

year_range = [i for i in range(1976, 1976 + 7)]
df['year'] = year_range * N

i_indicator = []
for i in range(1, 1 + 595):
  indv_i = [i]
  i_indicator += indv_i * T
df['i'] = i_indicator

df.head(14)

,exp,wks,bluecol,ind,south,smsa,married,gender,union,edu,col,lwage,year,i
0,3,32,0,0,1,0,1,0,0,9,0,5.56068,1976,1
1,4,43,0,0,1,0,1,0,0,9,0,5.72031,1977,1
2,5,40,0,0,1,0,1,0,0,9,0,5.99645,1978,1
3,6,39,0,0,1,0,1,0,0,9,0,5.99645,1979,1
4,7,42,0,1,1,0,1,0,0,9,0,6.06146,1980,1
5,8,35,0,1,1,0,1,0,0,9,0,6.17379,1981,1
6,9,32,0,1,1,0,1,0,0,9,0,6.24417,1982,1
7,30,34,1,0,0,0,1,0,0,11,0,6.16331,1976,2
8,31,27,1,0,0,0,1,0,0,11,0,6.21461,1977,2
9,32,33,1,1,0,0,1,0,1,11,0,6.26340,1978,2




```
# Dit is opgemaakt als code
```

#Pre-Assignment (1) - Regression for each year

## Manual

Estimate beta

In [ ]:
beta_hats = {}
beta_se = {}
beta_t = {}
for year in range(1976, 1976 + 7):
  df_year = df[df['year'] == year].drop(columns=['year', 'i'])
  y = df_year['lwage'].to_numpy()
  X = df_year.drop(columns=['lwage']).to_numpy()
  X = np.c_[np.ones(len(X)), X] #add constant
  beta_hats_year = np.linalg.inv(X.T @ X) @ X.T @ y
  u = y - X @ beta_hats_year
  N = X.shape[0]
  k = X.shape[1]
  var_u = (1/(N-k)) * (u.T @ u)
  beta_var_mat = var_u * np.linalg.inv(X.T @ X)
  beta_se_year = np.sqrt(np.diag(beta_var_mat))
  beta_t_year = beta_hats_year / beta_se_year

  beta_hats[f'beta_hats_{year}'] = beta_hats_year
  beta_se[f'beta_se_{year}'] = beta_se_year
  beta_t[f'beta_t_{year}'] = beta_t_year

In [ ]:
table_1_dict = {'variables': ['const'] + df.drop(columns = ['year', 'i', 'lwage']).columns.tolist()}

for year in range(1976, 1976+7):
    table_1_dict[f'{year} beta'] = beta_hats[f'beta_hats_{year}']
    table_1_dict[f'{year} se'] = beta_se[f'beta_se_{year}']
    table_1_dict[f'{year} t'] = beta_t[f'beta_t_{year}']

table_1 = pd.DataFrame(table_1_dict)
table_1_rounded = table_1.round(3)
table_1_rounded

,variables,1976 beta,1976 se,1976 t,1977 beta,1977 se,1977 t,1978 beta,1978 se,1978 t,...,1979 t,1980 beta,1980 se,1980 t,1981 beta,1981 se,1981 t,1982 beta,1982 se,1982 t
0,const,5.203,0.134,38.871,5.611,0.138,40.651,5.707,0.189,30.175,...,28.037,5.561,0.171,32.488,5.572,0.176,31.635,5.850,0.180,32.546
1,exp,0.010,0.001,8.698,0.008,0.001,7.102,0.008,0.001,5.743,...,5.081,0.006,0.001,4.894,0.006,0.001,4.462,0.005,0.001,3.661
2,wks,0.006,0.002,3.128,0.001,0.002,0.313,0.000,0.003,0.072,...,3.285,0.006,0.003,2.228,0.005,0.003,1.887,0.003,0.003,1.066
3,bluecol,-0.126,0.031,-4.140,-0.105,0.029,-3.668,-0.162,0.038,-4.283,...,-3.868,-0.167,0.035,-4.763,-0.131,0.036,-3.664,-0.165,0.037,-4.415
4,ind,0.020,0.025,0.797,0.018,0.023,0.755,0.044,0.031,1.445,...,2.147,0.087,0.027,3.173,0.091,0.028,3.194,0.090,0.029,3.065
5,south,-0.056,0.026,-2.108,-0.059,0.025,-2.378,-0.053,0.033,-1.627,...,-2.069,-0.054,0.029,-1.850,-0.055,0.030,-1.817,-0.058,0.031,-1.845
6,smsa,0.181,0.026,6.990,0.156,0.024,6.405,0.150,0.031,4.832,...,4.764,0.163,0.028,5.722,0.174,0.029,6.042,0.162,0.030,5.427
7,married,0.094,0.046,2.059,0.071,0.042,1.694,0.069,0.052,1.326,...,1.802,0.096,0.048,1.992,0.151,0.051,2.961,0.104,0.049,2.112
8,gender,-0.290,0.055,-5.283,-0.346,0.051,-6.827,-0.425,0.064,-6.637,...,-6.158,-0.365,0.058,-6.284,-0.264,0.062,-4.258,-0.318,0.061,-5.181
9,union,0.121,0.027,4.492,0.109,0.026,4.245,0.063,0.033,1.907,...,1.893,0.080,0.030,2.694,0.097,0.031,3.121,0.112,0.032,3.498


## With packages for checking accuracy of results

In [ ]:
OLS_by_year = {}
for year in range(1976, 1976 + 7):
  df_year = df[df['year'] == year]
  X = df_year.drop(columns=['year', 'i', 'lwage'])
  X = sm.add_constant(X)
  y = df_year['lwage']
  res = sm.OLS(y, X).fit() #non-robust specification
  OLS_by_year[f'ols_{year}'] = res

In [ ]:
for year in range(1976, 1976 + 7):
  print(f'Year {year}')
  print(OLS_by_year[f'ols_{year}'].summary())

Year 1976
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.489
Model:                            OLS   Adj. R-squared:                  0.479
Method:                 Least Squares   F-statistic:                     50.71
Date:                Tue, 24 Feb 2026   Prob (F-statistic):           1.03e-77
Time:                        14:00:54   Log-Likelihood:                -81.393
No. Observations:                 595   AIC:                             186.8
Df Residuals:                     583   BIC:                             239.4
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.2034      0.134     38.87

#Pre-Assignment (2) - Pooled

#(a) Between Estimation

##Manual

In [ ]:
N = df['i'].nunique()
T = df.groupby('i').size().iloc[0]
D = np.kron(np.identity(N), np.ones((T,1)))
P_D = D @ np.linalg.inv(D.T @ D) @ D.T
y = df['lwage'].to_numpy()
X = df.drop(columns=['lwage', 'year', 'i']).to_numpy()
X = np.c_[np.ones(len(X)), X] #add constant
y_avg = P_D @ y
X_avg = P_D @ X
beta_hats_btwn = np.linalg.inv(X_avg.T @ X_avg) @ X_avg.T @ y_avg
u_btwn = y_avg - X_avg @ beta_hats_btwn
k = X.shape[1]
var_u_btwn = (1/(N-k)) * sum(u_btwn**2)

#First unknown variance: variance of alpha
var_alpha_i = 1/T * var_u_btwn

#variance
var_matrix_beta_btwn = var_u_btwn * np.linalg.inv(X_avg.T @ X_avg)
beta_se_btwn = np.sqrt(np.diag(var_matrix_beta_btwn))
beta_t_btwn = beta_hats_btwn / beta_se_btwn

#R-squared of between estimation
total_sum_squares = sum((y_avg - y_avg.mean()) ** 2)
residual_sum_squares = sum((u_btwn) ** 2)
R_2_btwn = 1 - residual_sum_squares / total_sum_squares

In [ ]:
print("estimated variance of alpha_i using between estimation:")
print(f'{var_alpha_i}')

table_2 = pd.DataFrame({
    'variables': ['const'] + df.drop(columns = ['year', 'i', 'lwage']).columns.tolist(),
    'beta_hats': beta_hats_btwn,
    'se': beta_se_btwn,
    't': beta_t_btwn
})
table_2_rounded = table_2.round(3)
table_2_rounded


estimated variance of alpha_i using between estimation:
0.0757739767994883


,variables,beta_hats,se,t
0,const,5.263,0.207,25.381
1,exp,0.007,0.001,6.084
2,wks,0.010,0.004,2.751
3,bluecol,-0.176,0.035,-5.081
4,ind,0.064,0.026,2.433
5,south,-0.055,0.027,-2.068
6,smsa,0.170,0.026,6.470
7,married,0.135,0.049,2.769
8,gender,-0.300,0.056,-5.364
9,union,0.119,0.030,3.967


##with packages

In [ ]:
df_avg = df.groupby('i').mean()
y_bar = df_avg['lwage']
X_bar = df_avg.drop(columns=['year', 'lwage'])
X_bar = sm.add_constant(X_bar)
ols_btwn = sm.OLS(y_bar, X_bar).fit()
print(ols_btwn.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.512
Method:                 Least Squares   F-statistic:                     57.76
Date:                Tue, 24 Feb 2026   Prob (F-statistic):           6.46e-86
Time:                        14:04:15   Log-Likelihood:                -70.657
No. Observations:                 595   AIC:                             165.3
Df Residuals:                     583   BIC:                             218.0
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.2634      0.207     25.381      0.0

#(b) Within Estimation

##Manual

since we will be performing demeaning, time invariant variables will fall out, we first take a look at what these variables are

In [ ]:
#is a certain variable constant across time for all individuals?
time_inv_varb = []
for varb in df.drop(columns = ['lwage', 'year', 'i']).columns:
  t_invariant_each_i = []
  for i in range(1, 1 + 595):
    varb_i = df[df['i']==i][varb]
    if len(varb_i.unique()) == 1:
      t_invariant_each_i.append(True)
    else:
      t_invariant_each_i.append(False)

  if all(t_invariant_each_i):
    time_inv_varb.append(varb)

print(f'time invariant variables: {time_inv_varb}')

time invariant variables: ['gender', 'edu', 'col']


we need to remove the columns for constant, gender, edu, and col

In [ ]:
N = 595
T = 7
M_D = np.identity(N*T) - P_D
#redefine X and we don't add constant
X = df.drop(columns=['year', 'i', 'lwage'] + time_inv_varb)
X_column_order = X.columns #record order of columns (time variant variables)
X = X.to_numpy()
beta_hats_wthn = np.linalg.inv(X.T @ M_D @ X) @ X.T @ M_D @ y
wthn_resids = (M_D @ y) - (M_D @ X) @ beta_hats_wthn
k = X.shape[1]
var_eta = (1/(N * (T-1) - k)) * sum(wthn_resids ** 2)
var_matrix_beta_wthn = var_eta * np.linalg.inv(X.T @ M_D @ X)
beta_se_wthn = np.sqrt(np.diag(var_matrix_beta_wthn))
beta_t_wthn = beta_hats_wthn / beta_se_wthn

Results

In [ ]:
print("estimated variance of eta_i using within estimation:")
print(f'{var_eta}')
table_3 = pd.DataFrame({
    'variables': X_column_order,
    'beta_hats': beta_hats_wthn,
    'beta_se': beta_se_wthn,
    'beta_t': beta_t_wthn
})
table_3.round(3)


estimated variance of eta_i using within estimation:
0.023476664933713053


,variables,beta_hats,beta_se,beta_t
0,exp,0.097,0.001,81.099
1,wks,0.001,0.001,1.894
2,bluecol,-0.025,0.014,-1.790
3,ind,0.021,0.016,1.333
4,south,-0.003,0.035,-0.092
5,smsa,-0.044,0.020,-2.233
6,married,-0.030,0.019,-1.581
7,union,0.034,0.015,2.271


##with packages

In [ ]:
df_panel = df
df_panel = df_panel.set_index(['i', 'year'])
y_wthn = df_panel['lwage']
X_wthn = df_panel.drop(columns=['lwage'])
res_wthn = PanelOLS(y_wthn, X_wthn, entity_effects=True, drop_absorbed=True).fit()
print(res_wthn.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.6525
Estimator:                   PanelOLS   R-squared (Between):              0.4704
No. Observations:                4165   R-squared (Within):               0.6525
Date:                Thu, Feb 19 2026   R-squared (Overall):              0.4706
Time:                        17:57:27   Log-likelihood                    2228.8
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      836.08
Entities:                         595   P-value                           0.0000
Avg Obs:                       7.0000   Distribution:                  F(8,3562)
Min Obs:                       7.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             836.08
                            

/tmp/ipython-input-122724339.py:5: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

gender, edu, col

  res_wthn = PanelOLS(y_wthn, X_wthn, entity_effects=True, drop_absorbed=True).fit()


#(c) FGLS

I used the results from a and b for c.

In [ ]:
N = df['i'].nunique()
T = df.groupby('i').size().iloc[0]

# estimating the theta and sigma alpha
sigma_u_bar2 = var_u_btwn / T
sigma_alpha2 = max(0.0, sigma_u_bar2 - var_eta / T)
theta = 1.0 - np.sqrt(var_eta / (var_eta + T * sigma_alpha2))

# quasi demeaning
regressors = [c for c in df.columns if c not in ['i', 'year', 'lwage']]
df_re = df.copy()
for c in ['lwage'] + regressors:
    df_re[c + '_bar'] = df_re.groupby('i')[c].transform('mean')

y_star = (df_re['lwage'] - theta * df_re['lwage_bar']).to_numpy()
X_star = np.column_stack([
    np.ones(len(df_re)) * (1.0 - theta),
    np.column_stack([(df_re[c] - theta * df_re[c + '_bar']).to_numpy() for c in regressors])
])

beta_re_manual = np.linalg.solve(X_star.T @ X_star, X_star.T @ y_star)

# standard errors
resid_star = y_star - X_star @ beta_re_manual
n_obs, k_re = X_star.shape
sigma2_star = (resid_star @ resid_star) / (n_obs - k_re)
var_beta_re_manual = sigma2_star * np.linalg.inv(X_star.T @ X_star)
se_re_manual = np.sqrt(np.diag(var_beta_re_manual))
t_re_manual = beta_re_manual / se_re_manual

re_manual_table = pd.DataFrame({
    'variables': ['const'] + regressors,
    'beta_manual_RE': beta_re_manual,
    'se_manual_RE': se_re_manual,
    't_manual_RE': t_re_manual
})

print('theta =', theta)
print('sigma_eta^2 =', var_eta)
print('sigma_alpha^2 =', sigma_alpha2)
re_manual_table.round(4)


theta = 0.7896177282152672
sigma_eta^2 = 0.023476664933713053
sigma_alpha^2 = 0.07242016752324358


,variables,beta_manual_RE,se_manual_RE,t_manual_RE
0,const,4.4572,0.0987,45.1517
1,exp,0.0486,0.0011,45.8921
2,wks,0.0016,0.0008,2.0873
3,bluecol,-0.0565,0.0169,-3.3359
4,ind,0.0072,0.0176,0.4118
5,south,-0.0135,0.0272,-0.4952
6,smsa,-0.0224,0.0204,-1.0996
7,married,-0.0735,0.0234,-3.1396
8,gender,-0.3335,0.0528,-6.3175
9,union,0.0682,0.0174,3.9230


EStimating RE using the package and comparing it to our estimates

In [ ]:
panel = df.set_index(['i', 'year'])
y_pkg = panel['lwage']
X_pkg = sm.add_constant(panel[regressors])
re_pkg = RandomEffects(y_pkg, X_pkg).fit(cov_type='unadjusted')

print(re_pkg.summary)

compare = pd.DataFrame({
    'manual': beta_re_manual,
    'package': re_pkg.params.values,
    'abs_diff': np.abs(beta_re_manual - re_pkg.params.values)
}, index=['const'] + regressors)

compare

                        RandomEffects Estimation Summary                        
Dep. Variable:                  lwage   R-squared:                        0.3690
Estimator:              RandomEffects   R-squared (Between):             -0.6696
No. Observations:                4165   R-squared (Within):               0.4925
Date:                Thu, Feb 19 2026   R-squared (Overall):             -0.3543
Time:                        17:57:27   Log-likelihood                    752.39
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      220.78
Entities:                         595   P-value                           0.0000
Avg Obs:                       7.0000   Distribution:                 F(11,4153)
Min Obs:                       7.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             220.78
                            

,manual,package,abs_diff
const,4.457238,4.457821,5.827975e-04
exp,0.048585,0.048566,1.838135e-05
wks,0.001636,0.001637,3.093531e-07
bluecol,-0.056471,-0.056490,1.934097e-05
ind,0.007246,0.007249,2.552824e-06
south,-0.013467,-0.013487,2.041749e-05
smsa,-0.022436,-0.022398,3.811862e-05
married,-0.073482,-0.073486,4.069256e-06
gender,-0.333454,-0.333524,7.003419e-05
union,0.068204,0.068218,1.401221e-05


# (d) Hausman Test

> Blockquote toevoegen



hausman using earlier results.

In [ ]:
# dropping the time invariant regressors for FE
time_invariant = ['gender', 'edu', 'col']
fe_regs = [c for c in regressors if c not in time_invariant]

# renaming the variables: can be aligned later, now it looks a bit messsy still
b_fe = beta_hats_wthn
V_fe = var_matrix_beta_wthn

# re
idx_map = [(['const'] + regressors).index(c) for c in fe_regs]

b_re_sub = beta_re_manual[idx_map]
V_re_sub = var_beta_re_manual[np.ix_(idx_map, idx_map)]

b_diff = b_fe - b_re_sub
V_diff = V_fe - V_re_sub
V_diff = 0.5 * (V_diff + V_diff.T)  # symmetrize for numerics

H = float(b_diff.T @ np.linalg.pinv(V_diff) @ b_diff)
df_h = len(fe_regs)
pval = 1 - chi2.cdf(H, df_h)

print("Hausman H =", H)
print("df =", df_h)
print("p-value =", pval)


Hausman H = 6529.498562394387
df = 8
p-value = 0.0


Hausman test using package. RE and FE are estimated using the package, but the linearmodels package does not support a readyd hausman code. So it the formula is applied manually here:
I re estimated the FE using the packages. This can be cleaned later and aligned with earlier parts later, but fe_pkg was not defined yet.

In [ ]:
panel = df.set_index(['i','year'])

# set the time invariant regressors
time_invariant = [c for c in regressors if df.groupby('i')[c].nunique().max() == 1]
fe_regs = [c for c in regressors if c not in time_invariant]

# FE (RE i already did in previous part of c)
X_fe_pkg = sm.add_constant(panel[fe_regs])
fe_pkg = PanelOLS(panel['lwage'], X_fe_pkg, entity_effects=True).fit(cov_type='unadjusted')

b_fe = fe_pkg.params[fe_regs].values
b_re = re_pkg.params[fe_regs].values

V_fe = fe_pkg.cov.loc[fe_regs, fe_regs].values
V_re = re_pkg.cov.loc[fe_regs, fe_regs].values

b_diff = b_fe - b_re
V_diff = V_fe - V_re
V_diff = 0.5*(V_diff + V_diff.T)

H = float(b_diff.T @ np.linalg.pinv(V_diff) @ b_diff)
df_h = len(fe_regs)
pval = 1 - chi2.cdf(H, df_h)

print("Hausman H =", H)
print("df =", df_h)
print("p-value =", pval)



Hausman H = 6528.1035834428985
df = 8
p-value = 0.0


# (e) Two-way Fixed Effects Estimation

In [ ]:
panel = df.set_index(['i', 'year'])


# Question (e): Two-Way Fixed Effects (entity + time)

y = panel['lwage']
X = panel[fe_regs]

twfe_res = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
).fit()

print(twfe_res.summary)

# Table with coefficients, standard errors and t-stats
twfe_table = pd.DataFrame({
    "beta": twfe_res.params,
    "se": twfe_res.std_errors,
    "t": twfe_res.tstats
}).round(4)

display(twfe_table)

print("R-squared (model):", float(twfe_res.rsquared))
print("R-squared (within):", float(twfe_res.rsquared_within))


                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.0051
Estimator:                   PanelOLS   R-squared (Between):              0.0005
No. Observations:                4165   R-squared (Within):               0.0065
Date:                Thu, Feb 19 2026   R-squared (Overall):              0.0005
Time:                        17:57:28   Log-likelihood                    2250.7
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2.6211
Entities:                         595   P-value                           0.0106
Avg Obs:                       7.0000   Distribution:                  F(7,3557)
Min Obs:                       7.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             2.6211
                            

/tmp/ipython-input-1384813067.py:15: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

exp

  ).fit()


,beta,se,t
wks,0.0009,0.0006,1.5749
bluecol,-0.0221,0.0138,-1.5987
ind,0.0224,0.0155,1.4414
south,0.0023,0.0344,0.0665
smsa,-0.0432,0.0195,-2.2127
married,-0.0290,0.0191,-1.5212
union,0.0307,0.0150,2.0464


R-squared (model): 0.005131707838780253
R-squared (within): 0.006531689491006687


## Conclusion – Question (e): Two-Way Fixed Effects

The two-way fixed effects model controls for both individual-specific effects and time-specific effects. This means that all time-invariant unobserved individual characteristics and common macroeconomic shocks are accounted for.

After including both entity and time fixed effects, most explanatory variables lose statistical significance. In particular, variables such as weeks worked, blue-collar occupation, industry, region (south), and marital status are no longer statistically significant at the 5% level.

Only **union membership** and **living in an SMSA area** remain statistically significant. Union membership has a positive and statistically significant effect on wages, indicating that union members earn higher wages even after controlling for unobserved heterogeneity and time effects. The SMSA variable has a negative and significant coefficient, suggesting that, conditional on individual and time effects, living in an SMSA is associated with slightly lower wages in this specification.

The variable experience (exp) is fully absorbed by the fixed effects and therefore removed from the regression. This occurs because its variation is explained by the combination of individual and time effects.

Overall, the results suggest that once we control for both individual heterogeneity and time effects, only union membership robustly explains wage differences.
